# 09 — Advanced: A2A Server

**Stage 9 of the workshop (Production, extended).** Exposes a Strands agent as an HTTP service speaking the Agent-to-Agent (A2A) protocol — server half of a two-process pair.

## Problem

Getting from "works on my laptop" to something a team can rely on — deploying an agent as a standalone service other agents/apps can call, rather than an in-process function, is a deployment/production concern.

## Concept

A2A differs from MCP in what crosses the wire: MCP exposes *tools* to an agent; A2A exposes a *whole agent* to other agents/services as a standalone deployment. Each side is its own process with its own model, own tools, own lifecycle. That's the tradeoff: heavier to stand up than an in-process agent-as-tool, but the two agents no longer have to share a runtime, a language, or even a machine.

Same two-terminal shape as `06-mcp/1-mcp_calculator.py`: start the server first, then the client in a second terminal.

## Architecture

```
  A2AServer(agent=calculator_agent, host=127.0.0.1, port=9000)
         │
         ▼
  serves agent card + message endpoint over HTTP
         │
         │   (a2a_client.py, run separately, discovers this
         │    agent's card, then sends it a message)
         ▼
  agent (Agent + system_prompt, no tools)
  answers arithmetic questions concisely
```

## Before running the client side

This IS the server process. `4-a2a_client.py` must be run in a **separate terminal/notebook** and will connect to `http://127.0.0.1:9000` — start this server first and leave it running before executing the client's code.

## Step 1 — Model setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands.multiagent.a2a import A2AServer

model = get_model()


## Step 2 — Define the agent and wrap it as an A2A server

In [ ]:
agent = Agent(
    model=model,
    name="calculator_agent",
    description="Answers arithmetic questions.",
    system_prompt="You answer arithmetic questions concisely, with just the number.",
)

server = A2AServer(agent=agent, host="127.0.0.1", port=9000)


## Step 3 — Start serving

This call blocks — it runs the HTTP server. Run `4-a2a_client.py` in another terminal/notebook while this cell is running.

In [ ]:
print("A2A server starting on http://127.0.0.1:9000 — run 4-a2a_client.py in another terminal.")
server.serve()
